# Continual Learning System — MVP Setup & Verification


---
## 0 — Pre-flight System Check
Must pass before installing anything.

In [1]:
import sys, subprocess, platform

# ── Python version ────────────────────────────────────────────────────────────
py = sys.version_info
assert py >= (3, 10), f"Python 3.10+ required. Got {py.major}.{py.minor}"
print(f"✅  Python {py.major}.{py.minor}.{py.micro}")

# ── OS ───────────────────────────────────────────────────────────────────────
print(f"✅  OS: {platform.system()} {platform.release()}")

# ── CUDA driver check (nvcc or nvidia-smi) ───────────────────────────────────
try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap",
         "--format=csv,noheader"],
        stderr=subprocess.DEVNULL
    ).decode().strip()
    for line in smi.splitlines():
        name, mem, drv, cc = [x.strip() for x in line.split(",")]
        mem_gb = int(mem.replace(" MiB", "")) / 1024
        print(f"✅  GPU detected : {name}")
        print(f"    VRAM        : {mem_gb:.1f} GB  (need ≥54 GB for Qwen3-27B bf16)")
        print(f"    Driver      : {drv}")
        print(f"    Compute cap : {cc}")
        if mem_gb < 8:
            print(f"⚠️  VRAM is below the 54 GB minimum for Qwen3-27B bf16.")
            print("    Switch to Qwen3-27B 4-bit (~14 GB) or Qwen3-35B 4-bit (~18 GB).")
        else:
            headroom = mem_gb - 54
            print(f"    Headroom    : ~{headroom:.0f} GB free after model load — sufficient for LoRA + dual-judge")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("❌  nvidia-smi not found. Confirm NVIDIA drivers are installed and GPU is visible.")
    sys.exit(1)

# ── CUDA toolkit version ──────────────────────────────────────────────────────
try:
    nvcc = subprocess.check_output(["nvcc", "--version"], stderr=subprocess.DEVNULL).decode()
    cuda_ver = [l for l in nvcc.splitlines() if "release" in l][0].split("release ")[1].split(",")[0]
    print(f"✅  CUDA toolkit : {cuda_ver}")
    CUDA_VER = cuda_ver.replace(".", "")
except (subprocess.CalledProcessError, FileNotFoundError, IndexError):
    print("⚠️  nvcc not on PATH. Will infer CUDA version from PyTorch after install.")
    CUDA_VER = "124"   # assume 12.4 for RTX Blackwell; adjust if different

print(f"\nPre-flight passed. Proceeding with CUDA_VER={CUDA_VER}")

✅  Python 3.12.12
✅  OS: Linux 6.6.122+
✅  GPU detected : Tesla T4
    VRAM        : 15.0 GB  (need ≥54 GB for Qwen3-27B bf16)
    Driver      : 580.105.08
    Compute cap : 7.5
    Headroom    : ~-39 GB free after model load — sufficient for LoRA + dual-judge
✅  GPU detected : Tesla T4
    VRAM        : 15.0 GB  (need ≥54 GB for Qwen3-27B bf16)
    Driver      : 580.105.08
    Compute cap : 7.5
    Headroom    : ~-39 GB free after model load — sufficient for LoRA + dual-judge
✅  CUDA toolkit : 12.8

Pre-flight passed. Proceeding with CUDA_VER=128


---
## 1 - Install

In [2]:
import sys
import subprocess

def run(cmd):
    subprocess.run(cmd, check=True)

# 1) Ensure uv is available
try:
    run([sys.executable, "-m", "uv", "--version"])
except Exception:
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])

def uv_install(*args):
    run([sys.executable, "-m", "uv", "pip", "install", "--python", sys.executable, *args])

# 2) PyTorch + CUDA 12.4
uv_install(
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "torch==2.5.1",
    "torchvision==0.20.1",
    "torchaudio==2.5.1",
)

# 3) Core training / model stack
uv_install(
    "unsloth",
    "transformers",
    "peft",
    "trl",
    "accelerate",
    "bitsandbytes",
    "xformers",
    "causal-conv1d",
    "git+https://github.com/fla-org/flash-linear-attention.git"
)

# 4) Data / retrieval / utilities
uv_install(
    "datasketch",
    "detoxify",
    "datasets",
    "arxiv",
    "sentence-transformers",
    "safetensors",
    "huggingface_hub",
    "jsonlines",
    "tqdm",
    "numpy",
    "matplotlib",
    "scikit-learn",
)

# 5) Merge / eval / app stack
uv_install(
    "mergekit",
    "evaluate",
    "gradio",
    "fastapi",
    "uvicorn",
)

/usr/bin/python3: No module named uv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 62.9 MB/s eta 0:00:00


Using Python 3.12.12 environment at: /usr
Resolved 27 packages in 786ms
 Downloaded torchaudio
 Downloaded torchvision
 Downloaded nvidia-cuda-nvrtc-cu12
 Downloaded nvidia-cuda-cupti-cu12
 Downloaded nvidia-nvjitlink-cu12
 Downloaded triton
 Downloaded nvidia-curand-cu12
 Downloaded nvidia-cusolver-cu12
 Downloaded nvidia-nccl-cu12
 Downloaded nvidia-cusparse-cu12
 Downloaded sympy
 Downloaded nvidia-cufft-cu12
 Downloaded nvidia-cublas-cu12
 Downloaded nvidia-cudnn-cu12
 Downloaded torch
Prepared 17 packages in 31.22s
Uninstalled 17 packages in 2.06s
Installed 17 packages in 245ms
 - nvidia-cublas-cu12==12.8.4.1
 + nvidia-cublas-cu12==12.4.5.8
 - nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-cupti-cu12==12.4.127
 - nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-nvrtc-cu12==12.4.127
 - nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cuda-runtime-cu12==12.4.127
 - nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cudnn-cu12==9.1.0.70
 - nvidia-cufft-cu12==11.3.3.83
 + nvidia-cufft-cu12==11.2.1.3
 - 

In [3]:
import fla
import causal_conv1d
print("Fast kernels OK")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:1745: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.


Fast kernels OK


---
## 2 — HuggingFace Authentication
Qwen3-27B requires accepting the model license on HuggingFace.  
Log in now so model download in Phase 2 is non-interactive.

In [4]:
from huggingface_hub import get_token, whoami, login

# ── Check if already logged in ────────────────────────────────────────────────
token = get_token()

if token is not None:
    try:
        info = whoami(token=token)
        print(f"✅  Already logged in as: {info['name']}")
    except Exception:
        token = None

if token is None:
    print("No HuggingFace token found.")
    print("Run the cell below after pasting your token from https://huggingface.co/settings/tokens")
    print("Token needs READ access to gated repos (Qwen3-27B requires license acceptance).")
    print()
    print(">>> login(token='hf_YOUR_TOKEN_HERE')")
    print()
    print("Or run from terminal: huggingface-cli login")

# ── Check Qwen3-27B is accessible (metadata only, no download) ────────────────
from huggingface_hub import model_info
MODEL_ID = "Qwen/Qwen3.5-4B"
try:
    info = model_info(MODEL_ID)
    print(f"✅  {MODEL_ID} is accessible on HuggingFace Hub")
    print(f"    License: {info.card_data.get('license', 'check model card')}")
    print(f"    (Model will be downloaded in Phase 2 — not here)")
except Exception as e:
    print(f"⚠️  Cannot access {MODEL_ID}: {e}")
    print("    Ensure you are logged in and have accepted the Qwen3-27B license on the Hub.")

No HuggingFace token found.
Run the cell below after pasting your token from https://huggingface.co/settings/tokens
Token needs READ access to gated repos (Qwen3-27B requires license acceptance).

>>> login(token='hf_YOUR_TOKEN_HERE')

Or run from terminal: huggingface-cli login
✅  Qwen/Qwen3.5-4B is accessible on HuggingFace Hub
    License: apache-2.0
    (Model will be downloaded in Phase 2 — not here)


---
## 7 — Final Setup Checklist
All smoke tests aggregated into a single pass/fail summary.  
Every item must show ✅ before you write a single line of pipeline code.

In [5]:
import torch, importlib, subprocess, sys

print("=" * 70)
print("  CONTINUAL LEARNING MVP — FINAL SETUP CHECKLIST")
print("=" * 70)

checks = []

def check(label: str, fn):
    try:
        result = fn()
        status = "✅" if result else "❌"
        detail = "" if result else "  ← FAILED"
    except Exception as e:
        status = "❌"
        detail = f"  ← {type(e).__name__}: {e}"
        result = False
    checks.append(result)
    print(f"  {status}  {label}{detail}")

# ── System ────────────────────────────────────────────────────────────────────
print("\n  [ SYSTEM ]")
check("Python ≥ 3.10",
      lambda: sys.version_info >= (3, 10))
check("CUDA GPU visible",
      lambda: torch.cuda.is_available())
check("VRAM ≥ 54 GB (Qwen3-27B bf16 minimum)",
      lambda: torch.cuda.get_device_properties(0).total_memory / 1024**3 >= 54)
check("bf16 tensors on CUDA",
      lambda: bool(torch.tensor([1.0], dtype=torch.bfloat16).cuda()))

# ── Core ML Stack ─────────────────────────────────────────────────────────────
print("\n  [ CORE ML STACK ]")
for pkg in ["torch", "transformers", "peft", "trl", "accelerate", "bitsandbytes", "unsloth"]:
    check(f"{pkg} importable",
          lambda p=pkg: bool(importlib.import_module(p)))

# ── Data Pipeline ─────────────────────────────────────────────────────────────
print("\n  [ DATA PIPELINE ]")
for pkg, imp in [
    ("datasketch (MinHash Tier 1)",        "datasketch"),
    ("detoxify (Toxicity Tier 1)",         "detoxify"),
    ("datasets (HuggingFace)",             "datasets"),
    ("arxiv (Phase 2 data source)",        "arxiv"),
    ("faiss (Anchor vector index Tier 2)", "faiss"),
    ("sentence_transformers (Embeddings)", "sentence_transformers"),
]:
    check(pkg, lambda i=imp: bool(importlib.import_module(i)))

# ── Merge & Versioning ────────────────────────────────────────────────────────
print("\n  [ MERGE & VERSIONING ]")
for pkg in ["safetensors", "mergekit", "evaluate"]:
    check(f"{pkg} importable",
          lambda p=pkg: bool(importlib.import_module(p)))

# ── Logging & Utilities ───────────────────────────────────────────────────────
print("\n  [ LOGGING & UTILITIES ]")
for pkg in ["jsonlines", "yaml", "numpy", "matplotlib", "sklearn", "tqdm"]:
    check(f"{pkg} importable",
          lambda p=pkg: bool(importlib.import_module(p)))

# ── API & Demo Layer ──────────────────────────────────────────────────────────
print("\n  [ API & DEMO LAYER ]")
for pkg in ["gradio", "fastapi", "uvicorn"]:
    check(f"{pkg} importable",
          lambda p=pkg: bool(importlib.import_module(p)))

# ── HuggingFace Auth ──────────────────────────────────────────────────────────
print("\n  [ HUGGINGFACE AUTH ]")
from huggingface_hub import get_token
check("HF token present",
      lambda: get_token() is not None)

# ── Summary ───────────────────────────────────────────────────────────────────
passed = sum(checks)
total  = len(checks)
failed = total - passed

print()
print("=" * 70)
if failed == 0:
    print(f"  🟢  ALL {total} CHECKS PASSED — environment is ready for Phase 1, Day 1")
    print()
    print("  Next step: Phase 1, Step 01")
    print("  → Load Qwen3-27B bf16, verify 42 GB headroom, run inference test.")
else:
    print(f"  🔴  {failed} / {total} CHECKS FAILED")
    print("  Resolve all failures before proceeding.")
    print("  Re-run only the failed cells, then re-run this checklist cell.")
print("=" * 70)

  CONTINUAL LEARNING MVP — FINAL SETUP CHECKLIST

  [ SYSTEM ]
  ✅  Python ≥ 3.10
  ✅  CUDA GPU visible
  ❌  VRAM ≥ 54 GB (Qwen3-27B bf16 minimum)  ← FAILED
  ✅  bf16 tensors on CUDA

  [ CORE ML STACK ]
  ✅  torch importable
  ✅  transformers importable
  ✅  peft importable
  ✅  trl importable
  ✅  accelerate importable
  ✅  bitsandbytes importable
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth Zoo will now patch everything to make training faster!
  ✅  unsloth importable

  [ DATA PIPELINE ]
  ✅  datasketch (MinHash Tier 1)
  ✅  detoxify (Toxicity Tier 1)
  ✅  datasets (HuggingFace)
  ✅  arxiv (Phase 2 data source)
  ❌  faiss (Anchor vector index Tier 2)  ← ModuleNotFoundError: No module named 'faiss'
  ✅  sentence_transformers (Embeddings)

  [ MERGE & VERSIONING ]
  ✅  safetensors importable
  ✅  mergekit importable
  ✅  evaluate importable

  [ LOGGING & UTILITIES ]
  ✅  jsonlines importable
  ✅  yaml importable
  ✅  numpy importable
  ✅  matplotlib importable
  ✅  sklearn importable
  ✅  tqdm importable

  [ API & DEMO LAYER ]
  ✅  gradio importable
  ✅  fastapi importable
  ✅  uvicorn importable

  [ HUGGINGFACE AUTH ]
  ❌  HF token present  ← FAILED

  🔴  3 / 30 CHECKS FAILED
  Resolve all failures before proceeding.
  Re-run only the failed cells, then re-run this checklist cell.


In [6]:
import torch
print(torch.cuda.device_count())

2


---
# Base Model Loading & v0 Baseline Generation

In [7]:
# ==============================================================================
# BLOCK 01 — Base Model Initialization & v0 Baseline Generation
# ==============================================================================

import json
import torch
import os
from tqdm import tqdm
from unsloth import FastLanguageModel

MODEL_ID = "Qwen/Qwen3.5-4B"
MAX_SEQ_LENGTH = 2048
BENCHMARK_PATH = "/kaggle/input/datasets/nipung2010/anchor-bench/anchor_benchmark.json"
BASELINE_OUTPUT_PATH = "anchor_benchmark_v0_answers.json"
BATCH_SIZE = 6

print("=" * 60)
print("  BLOCK 01: LOADING BASE MODEL AND GENERATING v0 BASELINE")
print("=" * 60)

assert os.path.exists(BENCHMARK_PATH)
with open(BENCHMARK_PATH, "r", encoding="utf-8") as f:
    benchmark_data = json.load(f)

assert len(benchmark_data) == 50

# ------------------------------------------------
# Load model
# ------------------------------------------------
print(f"Loading {MODEL_ID} in 4-bit Quantization...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)


tokenizer = tokenizer.tokenizer

FastLanguageModel.for_inference(model)

model.eval()
torch.set_grad_enabled(False)
torch.backends.cuda.matmul.allow_tf32 = True

print("✅ Model and Tokenizer loaded successfully.")

# ------------------------------------------------
# Batched generation
# ------------------------------------------------
print("Generating v0 baseline answers for 50 anchor questions...")

v0_answers = []

for i in tqdm(range(0, len(benchmark_data), BATCH_SIZE),
              desc="Evaluating Anchor Benchmark"):

    batch = benchmark_data[i:i + BATCH_SIZE]

    # ---- build prompts ----
    prompts = []
    for item in batch:
        messages = [
            {"role": "system", "content": "You are a helpful, harmless, and honest AI assistant."},
            {"role": "user", "content": item["question"]},
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        prompts.append(prompt)

    # ---- tokenize batch ----
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to("cuda")

    input_lengths = inputs["attention_mask"].sum(dim=1)

    # ---- generate ----
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        use_cache=True,
        do_sample=False,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id,
    )

    # ---- decode ----
    for idx, item in enumerate(batch):

        start = input_lengths[idx].item()

        generated_tokens = outputs[idx][start:]

        response = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True,
        ).strip()

        v0_answers.append({
            "id": item["id"],
            "category": item["category"],
            "question": item["question"],
            "expected_keywords": item.get("expected_keywords", []),
            "v0_response": response,
        })

# ------------------------------------------------
# Save results
# ------------------------------------------------
print(f"Saving baseline to {BASELINE_OUTPUT_PATH}...")

with open(BASELINE_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(v0_answers, f, indent=2)

print("✅ Phase 1, Step 01 Complete.")

  BLOCK 01: LOADING BASE MODEL AND GENERATING v0 BASELINE
Loading Qwen/Qwen3.5-4B in 4-bit Quantization...
==((====))==  Unsloth 2026.5.6: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

✅ Model and Tokenizer loaded successfully.
Generating v0 baseline answers for 50 anchor questions...


Evaluating Anchor Benchmark: 100%|██████████| 9/9 [07:00<00:00, 46.69s/it]

Saving baseline to anchor_benchmark_v0_answers.json...
✅ Phase 1, Step 01 Complete.


---
###💻 Block 02: Tier 1 Filter (Statistical)

In [8]:
# ==============================================================================
# BLOCK 02 — Tier 1 Filter (MinHash + Detoxify)
# ==============================================================================
from datasketch import MinHash, MinHashLSH
from detoxify import Detoxify
import torch

def run_tier1_filter(raw_texts, threshold=0.85, toxicity_limit=0.05):
    print(f"Starting Tier 1 Filter: {len(raw_texts)} samples...")
    
    # 1. Deduplication
    lsh = MinHashLSH(threshold=threshold, num_perm=128)
    unique_texts = []
    
    for i, text in enumerate(raw_texts):
        m = MinHash(num_perm=128)
        for word in text.lower().split():
            m.update(word.encode('utf-8'))
        
        # If no match found in LSH, it's unique
        if not lsh.query(m):
            lsh.insert(f"idx_{i}", m)
            unique_texts.append(text)
            
    print(f"  ↳ Deduplication: Kept {len(unique_texts)} / {len(raw_texts)}")
    
    # 2. Toxicity Check
    print("  ↳ Running Toxicity Filter (GPU)...")
    tox_model = Detoxify('original', device='cuda')
    results = tox_model.predict(unique_texts)
    
    clean_texts = [
        txt for i, txt in enumerate(unique_texts) 
        if results['toxicity'][i] < toxicity_limit
    ]
    
    print(f"  ↳ Toxicity: Kept {len(clean_texts)} / {len(unique_texts)}")
    return clean_texts

raw_data = json.load(open("/kaggle/input/datasets/nipung2010/anchor-bench/raw_ai_data.json"))
clean_data = run_tier1_filter(raw_data)

Starting Tier 1 Filter: 94 samples...
  ↳ Deduplication: Kept 89 / 94
  ↳ Running Toxicity Filter (GPU)...
Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.1-alpha/toxic_original-c1212f89.ckpt" to /root/.cache/torch/hub/checkpoints/toxic_original-c1212f89.ckpt


100%|██████████| 418M/418M [00:01<00:00, 337MB/s]


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

  ↳ Toxicity: Kept 88 / 89


---
💻 Block 03: Tier 2 MVL Judge (Model-in-the-loop)

In [9]:
# ==============================================================================
# BLOCK 03A (MVL COMPLIANT) — Tier 2 Judge (PROMPT PRE-FILLING)
# ==============================================================================
import re
import os
import json
from tqdm import tqdm

print("=" * 60)
print("  BLOCK 03A: TIER 2 JUDGE & FLAG GENERATION")
print("=" * 60)

FastLanguageModel.for_inference(model)

auto_approved = []
flagged_for_review =[]
failed_parsing = 0

for i, text in enumerate(tqdm(clean_data, desc="Tier 2 Scoring")):
    messages =[
        {"role": "system", "content": "You are a strict data quality evaluator."},
        {"role": "user", "content": f"Rate the following text for factual accuracy and basic AI relevance.\nScore: 1 (Poor) to 5 (Excellent).\n\nText: {text}"}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # 🐛 THE MAGIC TRICK: Pre-fill the assistant's response to bypass the "Thinking Process"
    prompt += "The integer score is: "
    
    inputs = tokenizer(text=[prompt], return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens=3,  # Only need enough room for a single digit
        do_sample=False, 
        pad_token_id=tokenizer.eos_token_id
    )
    
    score_str = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    
    if i < 3:
        print(f"\n[DEBUG {i+1}] Text: '{text[:40]}...' | Model Output: '{score_str}'")
        
    match = re.search(r'[1-5]', score_str)
    if match:
        score = int(match.group())
        if score >= 4:
            auto_approved.append(text)
        elif score == 3:
            flagged_for_review.append(text)
    else:
        failed_parsing += 1

rejected_count = len(clean_data) - len(auto_approved) - len(flagged_for_review)

print(f"\n✅ Tier 2 Complete.")
print(f"   ↳ Auto-approved (Score 4-5): {len(auto_approved)}")
print(f"   ↳ Flagged for review (Score 3): {len(flagged_for_review)}")
print(f"   ↳ Auto-rejected (Score 1-2): {rejected_count - failed_parsing}")
print(f"   ↳ Failed parsing (Implicitly rejected): {failed_parsing}")

# Save auto-approved state
with open("tier2_auto_approved.json", "w", encoding="utf-8") as f:
    json.dump(auto_approved, f)

# Write flagged documents to a simple text file
REVIEW_FILE = "tier3_manual_review.txt"
with open(REVIEW_FILE, "w", encoding="utf-8") as f:
    for text in flagged_for_review:
        f.write(text.replace('\n', ' ') + "\n")

print(f"\n⚠️  ACTION REQUIRED BEFORE RUNNING NEXT CELL:")
print(f"   1. Open '{REVIEW_FILE}' from your Jupyter file browser.")
print(f"   2. Delete the entire line for any document you wish to REJECT.")
print(f"   3. Save the file, then run Block 03B.")

  BLOCK 03A: TIER 2 JUDGE & FLAG GENERATION


Tier 2 Scoring:   1%|          | 1/88 [00:14<20:50, 14.37s/it]


[DEBUG 1] Text: 'Artificial intelligence is a field focus...' | Model Output: '5.'


Tier 2 Scoring:   2%|▏         | 2/88 [00:14<08:49,  6.16s/it]


[DEBUG 2] Text: 'Machine learning lets models improve fro...' | Model Output: '5
The'


Tier 2 Scoring:   3%|▎         | 3/88 [00:15<04:59,  3.52s/it]


[DEBUG 3] Text: 'Deep learning uses layered neural networ...' | Model Output: '5.'


Tier 2 Scoring: 100%|██████████| 88/88 [00:48<00:00,  1.82it/s]


✅ Tier 2 Complete.
   ↳ Auto-approved (Score 4-5): 61
   ↳ Flagged for review (Score 3): 0
   ↳ Auto-rejected (Score 1-2): 27
   ↳ Failed parsing (Implicitly rejected): 0

⚠️  ACTION REQUIRED BEFORE RUNNING NEXT CELL:
   1. Open 'tier3_manual_review.txt' from your Jupyter file browser.
   2. Delete the entire line for any document you wish to REJECT.
   3. Save the file, then run Block 03B.


In [10]:
# ==============================================================================
# BLOCK 03B — Finalize Tier 3 Human Review
# ==============================================================================
import json
import os

print("=" * 60)
print("  BLOCK 03B: FINALIZE TIER 3 HUMAN REVIEW")
print("=" * 60)

# 1. Load Auto-Approved
with open("tier2_auto_approved.json", "r", encoding="utf-8") as f:
    approved_data = json.load(f)

# 2. Load Manual Review (only lines the user kept)
REVIEW_FILE = "tier3_manual_review.txt"
human_approved =[]

if os.path.exists(REVIEW_FILE):
    with open(REVIEW_FILE, "r", encoding="utf-8") as f:
        for line in f:
            clean_line = line.strip()
            if clean_line: # Ignore blank lines deleted by user
                human_approved.append(clean_line)

# 3. Combine
final_training_data = approved_data + human_approved

print(f"✅ Tier 3 Complete.")
print(f"   ↳ Auto-approved: {len(approved_data)}")
print(f"   ↳ Human-approved: {len(human_approved)}")
print(f"   ↳ FINAL TRAINING SET: {len(final_training_data)} samples")

if len(final_training_data) == 0:
    print("❌ CRITICAL: 0 samples passed Tier 2 & Tier 3. Cannot proceed to Block 04.")
    print("If this happened, the model is scoring everything as a 1 or 2. Your raw data may be too short/basic.")
else:
    # Override the approved_data variable so Block 04 can use it seamlessly
    approved_data = final_training_data
    print("🚀 Ready for Block 04 (Replay Mixing)!")

  BLOCK 03B: FINALIZE TIER 3 HUMAN REVIEW
✅ Tier 3 Complete.
   ↳ Auto-approved: 61
   ↳ Human-approved: 0
   ↳ FINAL TRAINING SET: 61 samples
🚀 Ready for Block 04 (Replay Mixing)!


In [11]:
# ==============================================================================
# BLOCK 04 (MVL COMPLIANT) — Flat Replay Buffer & Batching
# ==============================================================================
import os
import json
import random
import sys

print("=" * 60)
print("  BLOCK 04: REPLAY BUFFER MIXING (FLAT MVL VERSION)")
print("=" * 60)

REPLAY_PATH = "/kaggle/input/datasets/nipung2010/anchor-bench/replay_buffer_seed.json"

assert os.path.exists(REPLAY_PATH), f"❌ {REPLAY_PATH} missing."

# 1. Load Flat Replay Buffer (No Anchor logic, No Stratification)
with open(REPLAY_PATH, "r", encoding="utf-8") as f:
    flat_replay_buffer = json.load(f)

print(f"Loaded Flat Replay Buffer: {len(flat_replay_buffer)} total samples")

new_data_count = len(approved_data) # From Block 03 Tier 2 Judge
if new_data_count == 0:
    print("❌ No new data passed the Tier 2 Judge. Halting Cycle 1.")
    sys.exit(1)

# 2. Hardcoded MVL Ratio: 30% New Data, 70% Replay
replay_count = int((new_data_count / 0.30) * 0.70)

# Randomly sample from the flat buffer (using choices to allow replacement if buffer is small)
sampled_replay = random.choices(flat_replay_buffer, k=replay_count)

# 3. Combine and Shuffle
training_batch_texts = approved_data + sampled_replay
random.shuffle(training_batch_texts)

print(f"✅ Training Batch Ready: {len(training_batch_texts)} total samples.")
print(f"   ↳ {new_data_count} New Samples (Tier 2 Approved)")
print(f"   ↳ {replay_count} Replay Samples (Randomly drawn from flat buffer)")

  BLOCK 04: REPLAY BUFFER MIXING (FLAT MVL VERSION)
Loaded Flat Replay Buffer: 200 total samples
✅ Training Batch Ready: 203 total samples.
   ↳ 61 New Samples (Tier 2 Approved)
   ↳ 142 Replay Samples (Randomly drawn from flat buffer)


In [12]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

CUDA available: True
GPU: Tesla T4
Capability: (7, 5)
BF16 supported: False


In [13]:
# ==============================================================================
# BLOCK 05 — LoRA Training Loop (Unsloth)
# ==============================================================================
import gc
import torch
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

print("=" * 60)
print("  BLOCK 05: LORA TRAINING (Zone 2: Layers 8-24)")
print("=" * 60)

# 🛠️ MEMORY FIX 1: Aggressively clear residual memory from Block 03
gc.collect()
torch.cuda.empty_cache()

# 🛠️ MEMORY FIX 2: Enable Unsloth's Gradient Checkpointing & Gradients
torch.set_grad_enabled(True)
FastLanguageModel.for_training(model)

# 1. Format Data into ChatML structure for SFT
def format_prompts(examples):
    texts = []
    for text in examples["text"]:
        prompt = tokenizer.apply_chat_template([
                {"role": "system", "content": "You are a helpful, harmless, and honest AI assistant."},
                {"role": "user", "content": "Explain a useful fact, concept, or snippet of knowledge."},
                {"role": "assistant", "content": text}
            ],
            tokenize=False
        )
        texts.append(prompt)
    return {"formatted_text": texts}

dataset = Dataset.from_dict({"text": training_batch_texts})
dataset = dataset.map(format_prompts, batched=True)

# 2. Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    layers_to_transform = list(range(8, 25)), 
    bias = "none",
    use_gradient_checkpointing = "unsloth", # (This relies on for_training() above)
    random_state = 42,
)

# 3. Configure Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "formatted_text",
    max_seq_length = 1024,                   # 🛠️ MEMORY FIX 3: Reduced from 2048 to 1024
    dataset_num_proc = 2,
    packing = False, 
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        learning_rate = 1e-4,
        fp16 = True,
        bf16 = False,
        logging_steps = 1,
        optim = "paged_adamw_8bit",          # 🛠️ MEMORY FIX 4: Use 'paged' optimizer
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        num_train_epochs = 2,
        output_dir = "outputs",
        average_tokens_across_devices = False,
    ),
)

# 4. Train!
print("🚀 Starting LoRA Training...")
trainer_stats = trainer.train()

# 5. Save the Adapter (v1)
ADAPTER_SAVE_PATH = "adapter_v1"
model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
print(f"✅ Training Complete. Adapter saved to ./{ADAPTER_SAVE_PATH}")

  BLOCK 05: LORA TRAINING (Zone 2: Layers 8-24)


Map:   0%|          | 0/203 [00:00<?, ? examples/s]

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/203 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


🚀 Starting LoRA Training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 203 | Num Epochs = 2 | Total steps = 26
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 1,572,864 of 4,540,838,400 (0.03% trained)


Step,Training Loss
1,1.682208
2,1.747271
3,1.569848
4,1.888753
5,1.486468
6,1.822328
7,1.746288
8,1.790620
9,2.003629
10,1.929525


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-26/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in adapter_v1/tokenizer_config.json.


✅ Training Complete. Adapter saved to ./adapter_v1


In [14]:
# ==============================================================================
# BLOCK 06 (MVL COMPLIANT) — Evaluation & Binary Rollback Decision (BATCHED)
# ==============================================================================
import json
from tqdm import tqdm

print("=" * 60)
print("  BLOCK 06: EVALUATION & BINARY ROLLBACK DECISION")
print("=" * 60)

FastLanguageModel.for_inference(model)

EVAL_BATCH_SIZE = 6   # Safe for dual T4 4-bit Qwen 9B

# ------------------------------------------------------------------
# 1. Load v0 baseline and compute starting scores
# ------------------------------------------------------------------
with open(BASELINE_OUTPUT_PATH, "r", encoding="utf-8") as f:
    v0_data = json.load(f)

v0_scores = {"factual": 0, "reasoning": 0, "identity": 0}
category_counts = {"factual": 20, "reasoning": 20, "identity": 10}

for item in v0_data:
    passed_v0 = any(
        kw.lower() in item.get("v0_response", "").lower()
        for kw in item["expected_keywords"]
    )
    if passed_v0:
        v0_scores[item["category"]] += 1

v0_pct = {
    cat: (v0_scores[cat] / category_counts[cat]) * 100
    for cat in category_counts
}

print(
    f"Baseline (v0) -> "
    f"Factual: {v0_pct['factual']:.1f}%, "
    f"Reasoning: {v0_pct['reasoning']:.1f}%, "
    f"Identity: {v0_pct['identity']:.1f}%"
)

# ------------------------------------------------------------------
# 2. Batched Candidate Evaluation
# ------------------------------------------------------------------
print("\nEvaluating candidate model against Anchor Benchmark...")

candidate_scores = {"factual": 0, "reasoning": 0, "identity": 0}

for i in tqdm(
    range(0, len(benchmark_data), EVAL_BATCH_SIZE),
    desc="Candidate Inference"
):

    batch = benchmark_data[i:i + EVAL_BATCH_SIZE]

    prompts = []
    for item in batch:
        messages = [
            {"role": "system",
             "content": "You are a helpful, harmless, and honest AI assistant."},
            {"role": "user", "content": item["question"]},
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt)

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to("cuda")

    input_lengths = inputs["attention_mask"].sum(dim=1)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        repetition_penalty=1.15,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    # --------------------------------------------------------------
    # Decode each sample
    # --------------------------------------------------------------
    for idx, item in enumerate(batch):

        generated_tokens = outputs[idx][input_lengths[idx]:]

        response = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).lower()

        passed = any(
            kw.lower() in response
            for kw in item["expected_keywords"]
        )

        if passed:
            candidate_scores[item["category"]] += 1

# ------------------------------------------------------------------
# 3. Compute Candidate Percentages
# ------------------------------------------------------------------
candidate_pct = {
    cat: (candidate_scores[cat] / category_counts[cat]) * 100
    for cat in category_counts
}

print("\n--- Candidate Scores ---")
print(f"Factual:   {candidate_pct['factual']:.1f}%")
print(f"Reasoning: {candidate_pct['reasoning']:.1f}%")
print(f"Identity:  {candidate_pct['identity']:.1f}%")

# ------------------------------------------------------------------
# 4. MVL Binary Rollback Logic
# ------------------------------------------------------------------
identity_drop = v0_pct["identity"] - candidate_pct["identity"]
reasoning_drop = v0_pct["reasoning"] - candidate_pct["reasoning"]

system_state = "ACCEPT"
rollback_stage = 0

if identity_drop > 1.0 or reasoning_drop > 5.0:
    system_state = "REJECT"
    rollback_stage = 1

    print("\n❌ GATE FAILED: Catastrophic forgetting detected.")

    if identity_drop > 1.0:
        print(f"   ↳ Identity dropped by {identity_drop:.1f}% (>1%)")

    if reasoning_drop > 5.0:
        print(f"   ↳ Reasoning dropped by {reasoning_drop:.1f}% (>5%)")

    print("   ↳ ACTION: Adapter v1 discarded. Reverting to v0.")
else:
    print("\n✅ GATE PASSED: Model retained core knowledge safely.")

print("\nSystem State:", system_state)

  BLOCK 06: EVALUATION & BINARY ROLLBACK DECISION
Baseline (v0) -> Factual: 100.0%, Reasoning: 90.0%, Identity: 100.0%

Evaluating candidate model against Anchor Benchmark...


Candidate Inference: 100%|██████████| 9/9 [05:12<00:00, 34.69s/it]


--- Candidate Scores ---
Factual:   95.0%
Reasoning: 100.0%
Identity:  100.0%

✅ GATE PASSED: Model retained core knowledge safely.

System State: ACCEPT


In [15]:
# ==============================================================================
# BLOCK 07 (MVL COMPLIANT) — Merge & 5-Metric Logging
# ==============================================================================
import hashlib
import subprocess
import datetime
import os
import sys
import jsonlines

print("=" * 60)
print("  BLOCK 07: MERGE & MVL LOGGING")
print("=" * 60)

MERGED_DIR = "merged_model_v1"

if system_state == "ACCEPT":
    print("Executing DARE-TIES Merge (seed=42, density=0.70)...")
    
    # Free up memory before mergekit
    del model
    del trainer
    import torch
    torch.cuda.empty_cache()
    
    merge_cmd_python =[sys.executable, "-m", "mergekit.merge", "merge_spec.yaml", MERGED_DIR, "--copy-tokenizer", "--cuda", "--low-cpu-memory"]
    merge_cmd_cli =["mergekit-merge", "merge_spec.yaml", MERGED_DIR, "--copy-tokenizer", "--cuda", "--low-cpu-memory"]
    
    try:
        subprocess.run(merge_cmd_python, check=True)
        print(f"✅ Merge successful! New base model saved to ./{MERGED_DIR}")
    except subprocess.CalledProcessError:
        print("⚠️ Python module invocation failed. Falling back to CLI 'mergekit-merge'...")
        try:
            subprocess.run(merge_cmd_cli, check=True)
            print(f"✅ Merge successful! New base model saved to ./{MERGED_DIR}")
        except subprocess.CalledProcessError:
            print("❌ Merge failed entirely. Please check mergekit installation.")
            sys.exit(1)
else:
    print("Skipping merge due to Rollback.")

# Generate SHA of the adapter
adapter_file_path = os.path.join(ADAPTER_SAVE_PATH, "adapter_model.safetensors")
with open(adapter_file_path, "rb") as f:
    adapter_sha = hashlib.sha256(f.read()).hexdigest()

# Calculate overall Fast Score Delta (Average of the 3 categories for Cycle 1 simplicity)
overall_v0_score = sum(v0_pct.values()) / 3
overall_candidate_score = sum(candidate_pct.values()) / 3
fast_score_delta = overall_candidate_score - overall_v0_score

# 5 Metrics STRICTLY Required for Cycle 1 MVL (Section 23.1)
log_entry = {
    # Metadata
    "version": "v1",
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "adapter_sha256": adapter_sha,
    "system_state": system_state,
    
    # MVL 5 Metrics
    "replay_ratio": 0.70,                  # Hardcoded for Cycle 1
    "fast_score_delta": fast_score_delta,  # Consolidated into one metric
    "rollback_stage": rollback_stage,      # 0 or 1
    "kl_from_anchor": 0.0,                 # Placeholder, valid for MVL
    "domain_gain": candidate_pct['factual'] - v0_pct['factual'] # Using factual as proxy for domain gain in Cycle 1
}

with jsonlines.open("update_log.jsonl", mode="a") as writer:
    writer.write(log_entry)

print("\n✅ Cycle 1 logged successfully to update_log.jsonl")
print(f"🔒 Adapter Hash: {adapter_sha}")
print("🎉 MINIMUM VIABLE LOOP (MVL) FULLY SECURED AND COMPLETE! 🎉")

  BLOCK 07: MERGE & MVL LOGGING
Executing DARE-TIES Merge (seed=42, density=0.70)...


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


✅ Merge successful! New base model saved to ./merged_model_v1

✅ Cycle 1 logged successfully to update_log.jsonl
🔒 Adapter Hash: 9a1c34a3e11a5f4400c00103809207c0e55e13629a5a78e659a2282687bf9b3a
🎉 MINIMUM VIABLE LOOP (MVL) FULLY SECURED AND COMPLETE! 🎉


In [16]:
# ==============================================================================
# BLOCK 08 — Automated Version Control & Model Registry Push
# ==============================================================================
import os
import subprocess
import jsonlines
from huggingface_hub import HfApi

print("=" * 60)
print("  BLOCK 08: GITHUB & HUGGING FACE BACKUP")
print("=" * 60)

# --- Configuration ---
HF_REPO_ID = "NipunG2010/CLS-model" # ⚠️ CHANGE THIS
GITHUB_BRANCH = "main"

# 1. Generate a Data-Driven Commit Message
# We read the latest log entry to dynamically build the commit message
try:
    with jsonlines.open("update_log.jsonl", mode="r") as reader:
        logs = list(reader)
        latest_log = logs[-1]
        cycle_num = len(logs)
        
    state = latest_log["system_state"]
    f_delta = latest_log["fast_score_delta_factual"]
    r_delta = latest_log["fast_score_delta_reasoning"]
    i_delta = latest_log["fast_score_delta_identity"]
    sha_short = latest_log["adapter_sha256"][:8]

    commit_msg = (
        f"Cycle {cycle_num} [{state}]: "
        f"Factual {f_delta:+.1f}% | Reason {r_delta:+.1f}% | Identity {i_delta:+.1f}% "
        f"(Adapter: {sha_short})"
    )
    print(f"📝 Generated Commit Message:\n   \"{commit_msg}\"\n")
    
except Exception as e:
    print(f"⚠️ Could not read update_log.jsonl for commit message: {e}")
    commit_msg = f"Automated Cycle Backup - {datetime.datetime.now().strftime('%Y-%m-%d')}"

# 2. Push to Hugging Face (Adapters & Logs)
print("🚀 Pushing Adapter and Logs to Hugging Face...")
try:
    api = HfApi()
    
    # Create repo if it doesn't exist (set private=True to protect your moat!)
    api.create_repo(repo_id=HF_REPO_ID, private=True, exist_ok=True)
    
    # Upload the adapter weights
    api.upload_folder(
        folder_path=ADAPTER_SAVE_PATH,
        repo_id=HF_REPO_ID,
        path_in_repo=f"cycle_{cycle_num}_{state.lower()}/adapter",
        commit_message=commit_msg
    )
    
    # Upload the critical continual learning logs
    api.upload_file(
        path_or_fileobj="update_log.jsonl",
        path_in_repo="update_log.jsonl",
        repo_id=HF_REPO_ID,
        commit_message=commit_msg
    )
    api.upload_file(
        path_or_fileobj="replay_buffer_seed.json",
        path_in_repo=f"cycle_{cycle_num}_{state.lower()}/replay_buffer.json",
        repo_id=HF_REPO_ID,
        commit_message=commit_msg
    )
    print(f"✅ Successfully backed up artifacts to Hugging Face: https://huggingface.co/{HF_REPO_ID}")

except Exception as e:
    print(f"❌ Hugging Face push failed: {e}")
    print("   (Ensure you are logged in via `huggingface-cli login` and have write access).")

# 3. Push to GitHub (Code & Infrastructure)
print("\n🚀 Pushing Code & Configs to GitHub...")
try:
    # Git Add
    subprocess.run(["git", "add", "."], check=True, capture_output=True)
    
    # Git Commit
    commit_result = subprocess.run(["git", "commit", "-m", commit_msg], capture_output=True, text=True)
    if "nothing to commit" in commit_result.stdout:
        print("   ↳ No code changes to commit to GitHub.")
    else:
        # Git Push
        subprocess.run(["git", "push", "origin", GITHUB_BRANCH], check=True, capture_output=True)
        print(f"✅ Successfully pushed code to GitHub branch '{GITHUB_BRANCH}'.")

except subprocess.CalledProcessError as e:
    print(f"❌ GitHub push failed.")
    print(f"   Error: {e.stderr.decode('utf-8') if e.stderr else 'Unknown Git Error'}")
    print("   (Ensure 'git init' is run and 'origin' remote is set).")

print("=" * 60)
print("  CYCLE FULLY COMPLETED AND BACKED UP. READY FOR NEXT BATCH.")
print("=" * 60)

  BLOCK 08: GITHUB & HUGGING FACE BACKUP
⚠️ Could not read update_log.jsonl for commit message: 'fast_score_delta_factual'
🚀 Pushing Adapter and Logs to Hugging Face...
❌ Hugging Face push failed: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6a117b45-398baf4456bff8537eb0bfc4;dfb03f39-9c65-4044-8c6a-e29ea738afcd)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.
   (Ensure you are logged in via `huggingface-cli login` and have write access).

🚀 Pushing Code & Configs to GitHub...
❌ GitHub push failed.
   Error: fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).

   (Ensure 'git init' is run and 'origin' remote is set).
  CYCLE FULLY COMPLETED AND BACKED UP. READY FOR NEXT BATCH.
